# MongoDB Atlas와 Claude Managed Agents를 사용하는 사기 검토 agent

이 cookbook은 표준 CMA 패턴(custom tool과 MCP toolset)만 사용하고 **플랫폼 수준 MongoDB integration 없이**, [Claude Managed Agents](README.md)(CMA)에서 실행하는 agent에 **MongoDB Atlas**를 retrieval engine, graph store, system of record로 제공하는 방법을 보여 줍니다. 실행 가능한 helper는 이 notebook 옆의 [`mongodb_on_cma/`](mongodb_on_cma/)에 있습니다. 각 절에서 전체 파일을 연결하여 설명이 *MongoDB가 연결되는 곳*과 *agent가 가치를 더하는 곳*에 집중하도록 합니다.

**MongoDB가 연결되는 곳:** 대부분의 agent stack은 세 개나 네 개의 시스템을 조합합니다. semantic search용 vector database, keyword용 별도 engine, 관계용 graph store, 레코드 자체를 위한 operational database입니다. 각 접점마다 또 다른 integration, credential, agent의 세계관이 어긋날 수 있는 지점이 생깁니다. MongoDB는 이를 하나의 engine으로 합칩니다. 같은 문서를 **vector**(`$vectorSearch`), **full-text**(`$search`), reciprocal rank fusion 또는 RRF(`$rankFusion`)로 결합한 **hybrid**로 검색하고 **graph**(`$graphLookup`)로 탐색할 수 있습니다. 그리고 바로 그 문서를 agent가 읽고 쓰며 결정도 저장합니다. query language 하나, cluster 하나, 연결 하나(`pymongo`)입니다.

**agent가 가치를 더하는 곳:** CMA는 Anthropic 측에서 agent 루프와 sandbox를 실행합니다. 애플리케이션은 MongoDB로 이어지는 **data path**를 소유합니다. agent는 MongoDB가 검색한 내용을 추론하고, 위험한 호출에서는 CMA의 native `requires_action` gate로 사람을 기다리며, 판정을 다시 기록합니다. 이 모든 과정에서 database credential은 agent context나 sandbox에 들어가지 않습니다.

> 여기서 MongoDB는 애플리케이션이 제어하는 **operational data store와 retrieval engine**, 즉 system of record입니다. CMA가 플랫폼에서 native로 관리하는 memory 기능과는 구분되며 둘은 상호 보완적입니다.

**마치면 다음을 할 수 있습니다.**

- credential을 안전하게 유지하는 MongoDB Atlas data path를 세 방식으로 Claude Managed Agent에 연결하기.
- vector, full-text, hybrid, graph retrieval을 agent의 custom tool로 가져오기.
- 위험한 agent 결정에 CMA의 native human-in-the-loop 일시 정지 조건 두기.
- MongoDB Atlas cluster 하나를 agent의 system of record와 audit 기반으로 만들기.

실습 예제는 **human-in-the-loop 사기 검토 agent**이지만 패턴은 업종에 구애받지 않습니다. collection과 tool을 바꾸면 같은 형태가 지원, 조사, 운영 agent에 쓰입니다.

## 이 guide에서 다루는 내용

1. [managed agent에 MongoDB 연결하기](#1-connect-mongodb-to-a-managed-agent)
2. [Retrieve: 네 패턴, 하나의 engine](#2-retrieve-four-patterns-one-engine)
3. [전체 agent: human-in-the-loop 사기 검토](#3-the-end-to-end-agent-human-in-the-loop-fraud-review)
4. [system of record와 audit 기반으로서의 MongoDB Atlas](#4-mongodb-atlas-as-the-system-of-record-and-audit-backbone)
5. [되짚어 보기](#5-recap)


In [1]:
%%capture
# third-party dependency: Anthropic SDK(client.beta.*로 Claude Managed Agents 진행),
# pymongo(host 측 data path), python-dotenv, cryptography + pyjwt(AP2 mandate
# 서명/검증). PyPI package만 설치합니다. notebook은 인접한
# `mongodb_on_cma/` package와 `utilities.py`도 import하므로 repo clone에서 실행하세요(설정 참고).
%pip install -q "anthropic>=0.109.0" pymongo python-dotenv cryptography pyjwt

## 사전 요구 사항

**필요한 지식:** Python과 `pymongo` 기초, Claude API에 대한 기본적인 이해. MongoDB + Claude가 처음인가요? [library-RAG notebook](../third_party/MongoDB/rag_using_mongodb.ipynb)에서 `$vectorSearch`를 먼저 다룹니다. custom tool gate가 처음이라면 [`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb)이 왕복 과정을 설명합니다.

**필요한 도구:** Python 3.11+, [Anthropic API key](https://console.anthropic.com), MongoDB Atlas cluster.

## 설정

**필수:** `MONGO_URI`(Atlas SRV connection string)와 Anthropic API 접근. **무료 M0 cluster로 이 cookbook의 모든 것을 실행할 수 있습니다.** vector, full-text, hybrid `$rankFusion`, graph traversal 모두 가능하므로 유료 tier가 필요 없습니다. hybrid search는 MongoDB 8.0+가 필요한 native `$rankFusion` stage를 사용합니다. 현재 모든 Atlas cluster(M0 포함)는 8.0 이상이므로 기본적으로 충족됩니다.

```bash
uv sync --all-extras   # repo 루트에서
cp .env.example .env    # 그다음 MONGO_URI 추가(아직 설정하지 않았다면 Anthropic 인증도)
```

**선택적 embedding/rerank provider:** seed fixture는 사전 계산한 embedding을 제공하므로 provider 없이 실행됩니다. 실시간 embedding과 reranker 경로를 활성화하려면 `MDB_ATLAS_API_KEY`(`/v1/embeddings`와 `/v1/rerank`를 제공하는 MongoDB Atlas AI endpoint) 또는 `VOYAGE_API_KEY`(`voyageai` SDK) 중 **하나**를 설정하세요. `ENABLE_RERANK=1`은 두 번째 reranker 단계를 추가합니다. `AUTO_APPROVE=1`은 CI용으로 human gate를 결정적으로 처리합니다. `COOKBOOK_MODEL`은 agent model을 덮어씁니다(기본값 `claude-haiku-4-5`).

**아직 Atlas cluster가 없나요?** 관련 notebook [`rag_using_mongodb.ipynb`](../third_party/MongoDB/rag_using_mongodb.ipynb)은 무료 cluster 생성과 `MONGO_URI` 발급을 설명하며 [Atlas Search index 문서](https://www.mongodb.com/docs/atlas/atlas-search/create-index/)는 index 설정을 다룹니다.


학습용 코드는 **이 notebook 안에** 있습니다. 네 retrieval pipeline builder(2절), custom tool handler와 `requires_action` gate 루프(3절)입니다. notebook 옆의 [`mongodb_on_cma/`](mongodb_on_cma/) package는 읽기보다는 import하는 설정 boilerplate만 보관합니다. [`config.py`](mongodb_on_cma/config.py)(index 이름 + 조절값), [`embeddings.py`](mongodb_on_cma/embeddings.py)(Atlas/Voyage client), [`tools.py`](mongodb_on_cma/tools.py)(Atlas seed/index 설정 + 공유 decision/audit 문서 구성 함수), [`ap2_mandates.py`](mongodb_on_cma/ap2_mandates.py)(AP2 서명/검증 — `verify_mandates`로 호출하는 암호학 black box), [`seed.py`](mongodb_on_cma/seed.py)(예제 데이터 loader)입니다. 이들을 import하고 client를 연결하며 실행 설정을 결정합니다.


In [2]:
import hashlib
import json
import logging
import os
from datetime import UTC, datetime
from typing import Any

import dotenv
from anthropic import Anthropic
from mongodb_on_cma import (
    EMBED_DIM,
    build_audit_event,
    build_decision_doc,
    ensure_indexes,
    has_anthropic_auth,
    load_seed,
    make_embedding_client,
    missing_required_env,
    preflight,
    prepare_seed,
    rerank,
    resolve_model,
    seed_collection,
    server_version,
    supports_rank_fusion,
)
from mongodb_on_cma.ap2_mandates import (
    attach_mandates,
    store_mandate_receipt,
    tool_verify_mandates,
)
from pymongo import MongoClient
from utilities import wait_for_idle_status

dotenv.load_dotenv()

missing = missing_required_env()
assert not missing, f"Set these in your environment / .env: {missing}"
# Anthropic 인증은 유연하게 확인합니다. API key, auth token, profile(예: `ant`
# CLI의 workload-identity federation) 모두 작동하므로 아무것도 없으면 차단 대신 경고합니다.
if not has_anthropic_auth():
    print(
        "warning: no ANTHROPIC_API_KEY / ANTHROPIC_AUTH_TOKEN / ANTHROPIC_PROFILE found; "
        "relying on the SDK / `ant` CLI to resolve credentials."
    )

MODEL = resolve_model()
ENABLE_RERANK = os.getenv("ENABLE_RERANK", "").lower() in ("1", "true")
AUTO_APPROVE = os.getenv("AUTO_APPROVE", "").lower() in ("1", "true")

# ANTHROPIC_API_KEY가 profile/federation 자동 발견보다 우선한다는 SDK의 일회성
# 안내를 숨깁니다. API key 설정 시 예상되는 동작이며 `ant` / WIF 인증에서도 무해합니다.
logging.getLogger("anthropic.lib.credentials._auth").setLevel(logging.ERROR)
client = Anthropic()
mongo = MongoClient(os.environ["MONGO_URI"])
db = mongo["fraud_review_demo"]
coll = db["transactions"]
ai_client = make_embedding_client()  # Atlas / Voyage / None(seed는 사전 계산된 vector 제공)

print(
    f"model={MODEL}  rerank={'on' if ENABLE_RERANK else 'off'}  provider={'yes' if ai_client else 'none'}"
)

model=claude-haiku-4-5  rerank=on  provider=yes


## 1. managed agent에 MongoDB 연결하기

agent 루프와 sandbox는 **Anthropic-hosted**입니다. 이것이 "Managed"의 뜻입니다. 직접 host하는 것은 MongoDB로 가는 **data path**뿐입니다. 모든 경로에서 MongoDB credential은 경계의 *당신 측*에 있습니다. agent context, cloud sandbox environment, agent가 읽을 수 있는 파일에 절대 들어가지 않습니다.

> **이 cookbook이 Claude Agent SDK 대신 `client.beta.*`로 CMA와 통신하는 이유.**
> Managed Agents(CMA)는 [hosted REST API](https://platform.claude.com/docs/en/managed-agents/overview)입니다. Anthropic이 agent 루프와 sandbox를 실행하고, 표준 Anthropic SDK의 beta 인터페이스(`client.beta.agents`, `client.beta.sessions`, `client.beta.sessions.events`)로 agent/세션을 만들고 event를 streaming하여 진행합니다. 별도의 "CMA SDK"는 없습니다. *이것이* CMA 사용 방법입니다. [Claude Agent SDK](https://code.claude.com/docs/en/agent-sdk/overview)(`claude-agent-sdk`, repo의 [`claude_agent_sdk/`](../claude_agent_sdk/) cookbook에서 소개)는 다른 제품입니다. **자체 process 안에서**, 자체 infrastructure에서 agent 루프를 실행하는 library입니다. 둘은 상호 보완적입니다. Agent SDK로 로컬 prototype을 만들고 CMA에서 production을 실행할 수 있지만, 이와 같은 CMA integration은 beta sessions/events API를 기반으로 합니다. 이 선택 덕분에 아래 credential 경계도 작동합니다. *당신의* process가 각 custom tool 호출을 처리하므로 MongoDB secret은 host 측에 머뭅니다(Path A 참고).

data path를 연결하는 방식은 세 가지입니다. **이 cookbook은 내내 Path A를 사용합니다.** 권장되는 가장 가벼운 패턴이며 credential을 완전히 host 측에 유지합니다. 빠짐없이 소개하기 위해 이후 Path B와 C를 요약합니다.

### Path A: host 측 custom tool(권장)

애플리케이션이 `custom` tool을 정의합니다. agent가 호출하면 세션은 `requires_action`으로 일시 정지하고, **당신의** process가 `pymongo`로 query를 실행한 뒤 `user.custom_tool_result`를 게시합니다. secret은 sandbox에 전혀 들어가지 않고 MongoDB는 private VPC 안에 있을 수 있으며, agent는 고정된 audit 대상 query 집합으로 제한됩니다. 상시 server가 없는 가장 가벼운 경로이며 secret이 있는 모든 데이터 출처에 Anthropic이 권장하는 패턴입니다.

연결을 가리지 않고 보여 주기 위해 임시 `notes` collection의 작은 `find_notes` tool 하나로 전체 왕복 과정을 제시합니다. **이것이 핵심 integration입니다. custom tool 뒤에서 host 측으로 실행하는 몇 줄의 `pymongo`입니다.** 3절은 정확히 이 형태를 tool 다섯 개와 실제 작업으로 확장합니다.


In [3]:
notes = mongo["cookbook_mongodb_on_cma"]["notes"]
notes.delete_many({})
notes.insert_many(
    [
        {"note_id": "n1", "tag": "todo", "text": "Renew the SSL certificate before it expires."},
        {"note_id": "n2", "tag": "idea", "text": "Add a dark-mode toggle to the dashboard."},
        {"note_id": "n3", "tag": "todo", "text": "Email the quarterly report to the finance team."},
    ]
)

notes_agent = client.beta.agents.create(
    name="cookbook-mongodb-notes",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-with-mongodb-atlas"},
    model=MODEL,
    system=(
        "You answer questions about the user's notes. Use the find_notes tool to fetch notes "
        "(optionally filtered by `tag`) before answering, then give a concise final answer."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True, "permission_policy": {"type": "always_allow"}},
        },
        {
            "type": "custom",
            "name": "find_notes",
            "description": "Return stored notes, optionally filtered by tag (e.g. 'todo', 'idea').",
            "input_schema": {
                "type": "object",
                "properties": {"tag": {"type": "string", "description": "optional tag filter"}},
                "required": [],
            },
        },
    ],
)
notes_env = client.beta.environments.create(
    name="cookbook-mongodb-notes-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
notes_session = client.beta.sessions.create(
    environment_id=notes_env.id,
    agent={"type": "agent", "id": notes_agent.id, "version": notes_agent.version},
    title="MongoDB notes (host-side custom tool)",
)
print(f"seeded {notes.count_documents({})} notes; session: {notes_session.id}")

seeded 3 notes; session: sesn_016ufm58BVJ8aZeWjUt39bsq


event stream을 열고 질문을 보낸 뒤 왕복 과정을 진행합니다. 각 `agent.custom_tool_use`에서 **host 측** `pymongo`로 query를 실행하고 `user.custom_tool_result`를 게시합니다. 원본 MongoDB `ObjectId`는 JSON-serializable이 아니므로 결과를 `{"_id": 0}`으로 projection합니다.


In [4]:
def run_find_notes(args: dict) -> dict:
    """host 측 handler: pymongo는 여기서 실행합니다. 결과가 JSON 변환 가능하도록 _id(ObjectId)를 제외합니다."""
    flt = {"tag": args["tag"]} if args.get("tag") else {}
    return {"notes": list(notes.find(flt, {"_id": 0}))}


tool_use_events, responded, round_trips = {}, set(), []
question = "Using find_notes, list every note tagged 'todo', then tell me how many there are."

with client.beta.sessions.events.stream(notes_session.id) as stream:
    client.beta.sessions.events.send(
        session_id=notes_session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
    )
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded:
                        continue
                    tev = tool_use_events[event_id]
                    result = (
                        run_find_notes(tev.input)
                        if tev.name == "find_notes"
                        else {"error": "unknown"}
                    )
                    if tev.name == "find_notes":
                        round_trips.append(
                            {"tool_input": tev.input, "returned": len(result["notes"])}
                        )
                    client.beta.sessions.events.send(
                        session_id=notes_session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, notes_session.id)
print("host-side find_notes round-trips:")
for rt in round_trips:
    print(f"  find_notes({rt['tool_input']}) -> {rt['returned']} notes")

client.beta.sessions.archive(notes_session.id)
client.beta.environments.archive(notes_env.id)
client.beta.agents.archive(notes_agent.id)
notes.drop()
print("archived the demo agent / environment / session; dropped the notes collection")

host-side find_notes round-trips:
  find_notes({'tag': 'todo'}) -> 2 notes
archived the demo agent / environment / session; dropped the notes collection


### 다른 선택지(간략히)

Path A가 기본값이며 이 cookbook의 나머지도 이를 사용합니다. 맞지 않는 경우를 위한 대안이 둘 있습니다. 따라 하기 위해 필요하지는 않습니다.

- **Path B — sandbox 내 `pymongo`(self-hosted sandbox).** sandbox를 자체 infrastructure에서 실행하면 MongoDB client가 *안에* 있을 수 있고 agent는 일반 container environment variable인 `MONGO_URI`를 사용하여 `bash` tool에서 MongoDB를 직접 query합니다. self-hosted sandbox에서만 선택하세요(cloud sandbox에서는 connection string이 세션 이력에 들어감). 실행 가능한 image: [`self_hosted_sandboxes/docker/`](self_hosted_sandboxes/docker/).
- **Path C — self-hosted MongoDB MCP server.** 공식 MongoDB MCP server를 HTTPS 뒤에 host하고 `mcp_toolset`으로 등록하여 agent가 고정된 query 집합 대신 전체 인터페이스(`find`, `aggregate`, `$vectorSearch`)를 query하게 합니다. 표준 CMA MCP + vault 연결입니다.

장단점을 한 줄로 정리하면 **Path A**는 query 집합을 고정하고 secret을 backend에 유지하며(가장 안전한 기본값), **Path B**는 sandbox에 직접 접근을 주고, **Path C**는 server를 운영하는 대가로 agent에 가장 넓은 query 인터페이스를 제공합니다. 확신이 없으면 Path A를 유지하세요.


## 2. Retrieve: 네 패턴, 하나의 engine

agent가 근거를 얼마나 잘 갖추는지는 retrieval에 달려 있습니다. MongoDB retrieval 패턴 네 개를 **각각 독립적으로** 보여 줍니다. builder와 반환 내용을 보며 하나의 패턴을 자체 agent tool에 가져올 수 있습니다. 각각 일반 aggregation pipeline builder이며 3절의 agent는 tool로 같은 함수를 호출합니다.

| 패턴 | Builder | MongoDB stage |
| --- | --- | --- |
| Vector search | `build_vector_pipeline` | `$vectorSearch` |
| Full-text search | `build_lexical_pipeline` | `$search` |
| Hybrid(reciprocal rank fusion) | `build_rank_fusion_pipeline` | `$rankFusion` (8.0+) |
| Graph traversal | `build_graph_pipeline` | `$graphLookup` |

builder는 다음 셀 안에 정의되어 있습니다. 각각 자체 collection에 가져올 수 있는 aggregation stage 목록을 반환합니다. index 이름과 projection 필드는 [`config.py`](mongodb_on_cma/config.py)에서 가져오며, `EMBED_DIM`과 index 상수는 위 설정 셀에서 import합니다.

먼저 [`example_data/mongodb_on_cma/`](example_data/mongodb_on_cma/seed_transactions.jsonl)에서 seed fixture를 불러오고 index를 만듭니다. fixture가 사전 계산한 embedding을 제공하므로 provider key 없이 실행됩니다.


In [5]:
# 네 retrieval builder — 각각 자체 collection에 가져올 수 있는 일반 aggregation pipeline을
# 반환합니다. 3절의 agent가 custom tool로 호출하는 함수입니다.
from mongodb_on_cma.config import (
    DECIDED_STATUSES,
    LEXICAL_PATHS,
    PROJECT_FIELDS,
    SEARCH_INDEX_NAME,
    VECTOR_INDEX_NAME,
)


def _project_stage(with_score: bool = False) -> dict:
    proj: dict[str, Any] = {f: 1 for f in PROJECT_FIELDS}
    proj["_id"] = 0
    if with_score:
        proj["score"] = {"$meta": "score"}
    return {"$project": proj}


def build_vector_pipeline(
    qvec, *, limit, candidates=None, vector_index=VECTOR_INDEX_NAME, status_in=DECIDED_STATUSES
) -> list[dict]:
    return [
        {
            "$vectorSearch": {
                "index": vector_index,
                "path": "embedding",
                "queryVector": qvec,
                "numCandidates": candidates or max(50, limit * 10),
                "limit": limit,
                "filter": {"status": {"$in": list(status_in)}},
            }
        },
        _project_stage(),
    ]


def build_lexical_pipeline(query, *, limit, search_index=SEARCH_INDEX_NAME) -> list[dict]:
    return [
        {"$search": {"index": search_index, "text": {"query": query, "path": LEXICAL_PATHS}}},
        {"$limit": limit},
        _project_stage(),
    ]


def build_rank_fusion_pipeline(
    qvec,
    query,
    *,
    k,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> list[dict]:
    # $rankFusion(MongoDB 8.0+)은 두 input pipeline을 실행하고 reciprocal rank로 합칩니다.
    # 서버 측 aggregation 한 번입니다. 각 input pipeline의 weight는 1(동일)입니다.
    # semantic과 lexical의 비중을 바꾸려면 "combination": {"weights": {"vector": w, "lexical": w}}를 추가하세요.
    candidates = max(50, k * 10)
    per_branch = max(k * 4, 20)
    return [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vector": [
                            {
                                "$vectorSearch": {
                                    "index": vector_index,
                                    "path": "embedding",
                                    "queryVector": qvec,
                                    "numCandidates": candidates,
                                    "limit": per_branch,
                                    "filter": {"status": {"$in": list(status_in)}},
                                }
                            }
                        ],
                        "lexical": [
                            {
                                "$search": {
                                    "index": search_index,
                                    "text": {"query": query, "path": LEXICAL_PATHS},
                                }
                            },
                            {"$limit": per_branch},
                        ],
                    }
                },
            }
        },
        {"$limit": k},
        _project_stage(with_score=True),
    ]


def build_graph_pipeline(
    account_id: str, *, max_depth: int = 4, collection: str = "transactions"
) -> list[dict]:
    return [
        {"$match": {"sender.account_number": account_id}},
        {
            "$graphLookup": {
                "from": collection,
                "startWith": "$recipient.account_number",
                "connectFromField": "recipient.account_number",
                "connectToField": "sender.account_number",
                "as": "chain",
                "maxDepth": max_depth,
                "depthField": "depth",
            }
        },
    ]


def summarize_ring(graph_doc: dict, *, seed_account: str) -> dict:
    # $graphLookup chain을 fraud ring 신호로 바꿉니다. 시작 계정으로 돌아오는 순환 흐름,
    # layering(다수의 소액 송금), 전체 network 크기입니다.
    chain: list[dict] = list(graph_doc.get("chain", []))
    accounts: set[str] = set()
    small_transfers = 0
    circular_flow = False
    for edge in chain:
        sender = (edge.get("sender") or {}).get("account_number")
        recipient = (edge.get("recipient") or {}).get("account_number")
        accounts.update(a for a in (sender, recipient) if a)
        if recipient == seed_account:
            circular_flow = True
        if float(edge.get("amount", 0) or 0) < 1000:
            small_transfers += 1
    network_size = len(chain)
    layering = small_transfers >= 5
    return {
        "network_size": network_size,
        "unique_accounts": len(accounts),
        "circular_flow": circular_flow,
        "layering": layering,
        "suspicious_patterns": circular_flow or layering or network_size >= 3,
    }

In [6]:
docs = prepare_seed(load_seed(), now=datetime.now(UTC))
count = seed_collection(coll, docs)
print(f"seeded {count} transactions")

# 없으면 vector + Atlas Search index를 생성한 뒤 둘 다 query 가능하고
# 새로 넣은 문서를 실제로 반영할 때까지 대기합니다(Atlas Search는 eventually consistent).
ensure_indexes(coll, dim=EMBED_DIM)

check = preflight(coll)
for issue in check["issues"]:
    print("PREFLIGHT:", issue)
assert check["ok"], "Fix the preflight issues above before continuing."

# 이 cookbook은 hybrid search에 MongoDB 8.0+가 필요한 native `$rankFusion` stage를 사용합니다.
# 무료 M0 tier를 포함한 현재 모든 Atlas cluster는 8.0 이상이므로 기본적으로 충족됩니다.
# 이 확인은 MONGO_URI가 오래된 self-hosted server를 가리키면 명확하게 실패하게 할 뿐입니다.
version = server_version(coll)
assert supports_rank_fusion(version), (
    f"MongoDB {version} predates $rankFusion (needs 8.0+). Use an Atlas cluster (any tier) or a "
    f"self-hosted MongoDB 8.0+."
)
print(f"server={version}  hybrid search=$rankFusion (native)")

seeded 20 transactions
server=8.0.27  hybrid search=$rankFusion (native)


### 패턴 1: Vector search(`$vectorSearch`)

Semantic recall: `embedding`이 query vector에 가장 가까운 문서를 찾습니다. 이 예제는 기존 문서의 embedding을 query로 재사용하므로(판정된 선례를 찾는 대기 중 사례) embedding API 호출이 필요 없습니다. builder는 `status` 필드로 판정된 사례를 filtering합니다.


In [7]:
query_doc = coll.find_one({"transaction_id": "txn-review-struct"})  # 대기 중 사례
hits = list(coll.aggregate(build_vector_pipeline(query_doc["embedding"], limit=3)))
print("nearest decided precedents:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

nearest decided precedents:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### 패턴 2: Full-text search(`$search`, BM25)

text 필드의 keyword 및 구문 matching입니다. embedding이 흐릿하게 만드는 정확한 이름, ID, 코드를 잡습니다.


In [8]:
hits = list(
    coll.aggregate(
        build_lexical_pipeline("cash deposit just under the reporting threshold", limit=3)
    )
)
print("full-text matches:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

full-text matches:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-review-struct: Cash deposit of 4950 USD, just under the 5000 reporting threshold, follo
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account


### 패턴 3: Hybrid(reciprocal rank fusion)

vector와 full-text ranking을 하나의 결과 집합으로 결합합니다. `$rankFusion`(MongoDB 8.0+)은 두 input pipeline을 서버 측에서 실행하고 reciprocal rank로 합칩니다. client 측 merge와 두 번째 왕복 호출이 없습니다.


In [9]:
qvec, query = query_doc["embedding"], "structuring: cash deposit just under the threshold"
hits = list(coll.aggregate(build_rank_fusion_pipeline(qvec, query, k=3)))
print("hybrid via $rankFusion (server-side):")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

hybrid via $rankFusion (server-side):
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### 패턴 4: Graph traversal(`$graphLookup`)

`sender.account_number -> recipient.account_number` 연결을 따라 network를 드러냅니다(여기서는 순환 자금 흐름 고리). 이는 위의 유사도 ranking과 의도적으로 분리한 *관계* 신호입니다.


In [10]:
graph_doc = next(
    iter(coll.aggregate(build_graph_pipeline("ACC-RING-A", collection=coll.name))), {"chain": []}
)
ring = summarize_ring(graph_doc, seed_account="ACC-RING-A")
print(
    f"graph traversal from ACC-RING-A: network_size={ring['network_size']} "
    f"circular_flow={ring['circular_flow']} suspicious={ring['suspicious_patterns']}"
)

graph traversal from ACC-RING-A: network_size=4 circular_flow=True suspicious=True


**자신의 domain에 맞게 적용하세요.** 각 builder는 일반적인 형태입니다. 어떤 model로든 text를 embed하고 `build_vector_pipeline`을 해당 필드에 연결하세요. full-text의 검색 경로를 설정하고, `$rankFusion` input weight를 조절하여 semantic과 lexical의 비중을 바꾸고, graph `connectFromField`/`connectToField`를 관계(인용, 조직도, 공급망)에 맞게 변경하세요. 네 개 모두 위 셀에 정의되어 있으므로 자체 프로젝트에 바로 가져오세요.


## 3. 전체 agent: human-in-the-loop 사기 검토

이제 모두 함께 작동합니다. Managed Agent는 표시된 transaction을 검토하면서 각 추천을 2절의 MongoDB 신호(Path A 왕복 과정의 **custom tool**로 제공)에 근거하여 만들고 위험한 사례에서는 사람을 기다립니다. 결정과 append-only audit trail은 같은 cluster에 다시 기록됩니다.

<p align="center">
  <img src="../images/cma_with_mongodb_atlas.jpg" width="70%" alt="agent 루프는 Anthropic에서, data path는 notebook에서 실행됩니다. MongoDB Atlas는 retrieval 신호를 제공하고 결정과 audit trail을 저장합니다." />
</p>

**agent 루프는 Anthropic에서**, **data path는 notebook에서 실행됩니다**. 따라서 `MONGO_URI`와 embedding key는 agent나 sandbox에 들어가지 않습니다. 사람 대기는 지속적으로 유지됩니다. 응답할 때까지 `requires_action`으로 서버 측 세션이 idle에 머뭅니다. 아래의 간단한 streaming 루프는 개발용이며 production에서는 webhook 패턴의 지속 가능한 backend로 진행하세요.

### AP2 mandate: 검증한 뒤 추론하기

이 agent는 *agent가 시작한* 결제를 검토하므로 행동 분석 전에 사용자가 결제를 승인했음을 증명하는 암호학적으로 서명된 credential인 **AP2(Agent Payments Protocol)** mandate를 검증합니다. AP2는 agent 주도 상거래를 위한 [open protocol](https://github.com/google-agentic-commerce/AP2)입니다. mandate(**Checkout Mandate**와 **Payment Mandate**)는 변조를 식별할 수 있는 서명된 digital object로 audit 가능한 chain을 이룹니다. 이 cookbook을 따르기 위해 protocol을 배울 필요는 없습니다. **`verify_mandates` tool을 호출하고 판정에 따라 행동하세요**(valid, constraints-satisfied, double-spend). 서명과 검증의 세부 사항인 ES256 JWT는 [`ap2_mandates.py`](mongodb_on_cma/ap2_mandates.py)에 encapsulate되어 있으며, 서명된 mandate와 receipt는 transaction과 함께 MongoDB에 저장됩니다.

아래 셀은 "Trusted Surface" 역할을 합니다(keypair를 생성하고 대기 중 사례에 서명된 mandate 연결). private key는 로컬 Python 상태에만 남으며 agent, tool 결과, database에는 절대 들어가지 않습니다.


In [11]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

ts_private_key = ec.generate_private_key(ec.SECP256R1())  # Trusted Surface key, 실행마다 새로 생성
AGENT_PK = (
    ts_private_key.public_key()
    .public_bytes(serialization.Encoding.X962, serialization.PublicFormat.UncompressedPoint)
    .hex()
)
ts_public_key = ts_private_key.public_key()

PENDING = [
    "txn-review-clean",
    "txn-review-fraud",
    "txn-review-struct",  # AUTO_APPROVE에서 초기 데이터의 human override 사례
    "txn-review-high",
    "txn-review-ring",
]
mandates = attach_mandates(coll, PENDING, agent_pk=AGENT_PK, ts_private_key=ts_private_key)
print(f"attached AP2 mandates to {len(mandates)} transactions  agent_pk={AGENT_PK[:16]}…")

attached AP2 mandates to 5 transactions  agent_pk=04e1e32b7414d6f7…


### custom tool과 host 측 handler

각 tool은 `type: "custom"` tool입니다. agent가 하나를 호출하면 세션은 일시 정지하고 **이 notebook**이 `pymongo`로 handler를 실행하여 JSON 결과를 돌려보냅니다. agent는 database나 connection string에 전혀 접근하지 않습니다. handler 구현은 다음 셀에 정의되어 있으며 2절의 builder와 AP2 검증(`verify_mandates`)을 감쌉니다. `HANDLERS`는 다섯 data tool만 mapping합니다. gate 루프가 handler 대신 human resolver로 보내므로 `escalate`는 의도적으로 없습니다(다음 절 참고).


In [12]:
# host 측 tool 구현 — 각 custom tool 뒤의 pymongo 작업입니다. sandbox가 아닌
# 이 process에서 실행하며 2절의 builder를 재사용합니다. dispatch 연결 및
# AP2/record-decision wrapper는 다음 셀에 있습니다. build_decision_doc / build_audit_event
# (위에서 import)는 저장할 문서를 구성하며 AP2 module과 공유합니다.


def _jsonable(value):
    """Mongo/BSON 값(ObjectId, datetime, ...)을 재귀적으로 JSON에 안전한 타입으로 변환합니다."""
    if isinstance(value, dict):
        return {k: _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _pick_fields(candidate):
    keep = ("transaction_id", "text", "amount", "sender", "recipient", "decision", "score")
    return _jsonable({k: candidate[k] for k in keep if k in candidate})


def rerank_pool_size(k: int, fanout: int, *, enable_rerank: bool) -> int:
    return k * fanout if enable_rerank else k


def merge_rerank(candidates, rerank_results, *, top_k) -> list[dict]:
    def _score(r):
        s = getattr(r, "relevance_score", None)
        return (
            (s if s is not None else r.get("relevance_score", float("-inf")))
            if isinstance(r, dict)
            else (s if s is not None else float("-inf"))
        )

    seen: set[str] = set()
    out: list[dict] = []
    for result in sorted(rerank_results, key=_score, reverse=True):
        index = getattr(result, "index", None)
        if index is None and isinstance(result, dict):
            index = result.get("index")
        score = getattr(result, "relevance_score", None)
        if score is None and isinstance(result, dict):
            score = result.get("relevance_score")
        if index is None or index < 0 or index >= len(candidates):
            continue
        candidate = candidates[index]
        cid = str(candidate.get("transaction_id", index))
        if cid in seen:
            continue
        seen.add(cid)
        out.append({**candidate, "score": score})
        if len(out) >= top_k:
            break
    return out


def tool_get_transaction(coll, transaction_id) -> dict:
    doc = coll.find_one({"transaction_id": transaction_id})
    if not doc:
        return {"error": "not_found", "transaction_id": transaction_id}
    return _jsonable({k: v for k, v in doc.items() if k not in ("embedding", "_id")})


def tool_hybrid_search_similar_frauds(
    coll,
    transaction_id,
    k,
    *,
    enable_rerank=False,
    fanout=5,
    reranker=None,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    status_in=DECIDED_STATUSES,
) -> dict:
    txn = coll.find_one({"transaction_id": transaction_id})
    if not txn:
        return {"error": "not_found", "transaction_id": transaction_id, "similar": []}
    qvec = txn["embedding"]
    query = txn.get("text", "")
    pool_k = rerank_pool_size(k, fanout, enable_rerank=enable_rerank)
    candidates = list(
        coll.aggregate(
            build_rank_fusion_pipeline(
                qvec,
                query,
                k=pool_k,
                vector_index=vector_index,
                search_index=search_index,
                status_in=status_in,
            )
        )
    )
    candidates = [c for c in candidates if str(c.get("transaction_id")) != str(transaction_id)]
    if enable_rerank and reranker is not None and candidates:
        results = reranker(query, [c.get("text", "") for c in candidates], top_k=k)
        candidates = merge_rerank(candidates, results, top_k=k)
    else:
        candidates = candidates[:k]
    return {"similar": [_pick_fields(c) for c in candidates]}


def tool_detect_fraud_ring(coll, account_id, *, max_depth=4) -> dict:
    pipeline = build_graph_pipeline(account_id, max_depth=max_depth, collection=coll.name)
    docs = list(coll.aggregate(pipeline))
    return _jsonable(summarize_ring(docs[0] if docs else {"chain": []}, seed_account=account_id))


def tool_record_decision(
    db,
    transaction_id,
    decision,
    *,
    confidence,
    risk_factors,
    reasoning,
    reviewed_by,
    escalated=False,
    recommended_decision=None,
) -> dict:
    decision_doc = build_decision_doc(
        transaction_id,
        decision,
        confidence=confidence,
        risk_factors=risk_factors,
        reasoning=reasoning,
        reviewed_by=reviewed_by,
    )
    db["transaction_decisions"].insert_one(decision_doc)
    if escalated:
        audit = build_audit_event(
            "escalated_to_human",
            transaction_id,
            decision_id=decision_doc["decision_id"],
            severity="warning",
            event_data={"human_decision": decision, "recommended_decision": recommended_decision},
        )
    else:
        audit = build_audit_event(
            "decision_stored", transaction_id, decision_id=decision_doc["decision_id"]
        )
    db["audit_events"].insert_one(audit)
    # lifecycle status를 과거형(approve -> approved)으로 진행시켜 판정된 사례를
    # seed 용어 및 DECIDED_STATUSES와 맞추고 이후 검토에서 선례로 사용할 수 있게 합니다.
    status = {"approve": "approved", "reject": "rejected"}.get(decision, decision)
    db["transactions"].update_one({"transaction_id": transaction_id}, {"$set": {"status": status}})
    return {"recorded": True, "decision_id": decision_doc["decision_id"]}

In [13]:
TOOLS = [
    {
        "type": "custom",
        "name": "verify_mandates",
        "description": "Validate the AP2 Checkout and Payment Mandate JWTs (signature, constraints, "
        "double-spend). Run this FIRST. If valid=false, constraints_satisfied=false, or "
        "double_spend_detected=true, reject immediately.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "get_transaction",
        "description": "Fetch the full transaction record under review.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "hybrid_search_similar_frauds",
        "description": "Retrieve the most similar prior (already-decided) cases as precedent, using "
        "hybrid vector + full-text search.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "k": {"type": "integer", "description": "how many precedents (default 5)"},
            },
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "detect_fraud_ring",
        "description": "Trace the account's sender->recipient chain for circular-flow / mule / layering patterns.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "type": "custom",
        "name": "record_decision",
        "description": "Persist the final approve/reject decision with reasoning and an audit event.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "risk_factors": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "escalated": {"type": "boolean"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
            },
            "required": ["transaction_id", "decision", "reasoning"],
        },
    },
    {
        "type": "custom",
        "name": "escalate",
        "description": "Send a risky case to a human reviewer for the final decision. Use for "
        "medium-confidence, high-value, structuring, or fraud-ring cases.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["transaction_id", "recommended_decision", "reason"],
        },
    },
]

In [14]:
# host 측 handler — 각 data tool을 handler 함수에 mapping합니다.
reranker = (
    (lambda q, ds, top_k: rerank(q, ds, client=ai_client, top_k=top_k))
    if (ENABLE_RERANK and ai_client)
    else None
)


def _verify_mandates(inp):
    result = tool_verify_mandates(db, inp["transaction_id"], ts_public_key)
    ok = result["valid"] and result["constraints_satisfied"] and not result["double_spend_detected"]
    print(f"\n  [verify_mandates] {inp['transaction_id']}: {'pass' if ok else 'FAIL'}")
    return result


def _hybrid(inp):
    return tool_hybrid_search_similar_frauds(
        coll,
        inp["transaction_id"],
        inp.get("k", 5),
        enable_rerank=ENABLE_RERANK,
        reranker=reranker,
    )


def _record(inp):
    result = tool_record_decision(
        db,
        inp["transaction_id"],
        inp["decision"],
        confidence=inp.get("confidence", 0),
        risk_factors=inp.get("risk_factors", []),
        reasoning=inp.get("reasoning", ""),
        reviewed_by="human" if inp.get("escalated") else "agent",
        escalated=inp.get("escalated", False),
        recommended_decision=inp.get("recommended_decision"),
    )
    if inp["decision"] == "approve":  # 이후 double-spend 감지에 사용하는 AP2 receipt 저장
        txn_doc = coll.find_one(
            {"transaction_id": inp["transaction_id"]},
            {"checkout_mandate_jwt": 1, "mandate_id": 1, "agent_pk": 1},
        )
        if txn_doc and txn_doc.get("mandate_id"):
            checkout_hash = hashlib.sha256(txn_doc["checkout_mandate_jwt"].encode()).hexdigest()
            store_mandate_receipt(
                db, txn_doc["mandate_id"], txn_doc["agent_pk"], checkout_hash, "approve"
            )
    return result


HANDLERS = {
    "verify_mandates": _verify_mandates,
    "get_transaction": lambda inp: tool_get_transaction(coll, inp["transaction_id"]),
    "hybrid_search_similar_frauds": _hybrid,
    "detect_fraud_ring": lambda inp: tool_detect_fraud_ring(coll, inp["account_id"]),
    "record_decision": _record,
}

### agent, environment, session 생성하기

`model`, `system`, `tools`는 **agent**에 있습니다. 세션은 이를 참조하고 sandbox를 마련합니다. networking은 `limited`입니다. agent는 host 측 왕복 과정으로만 MongoDB에 접근합니다.


In [15]:
SYSTEM = """You are a financial fraud reviewer. You will be given transaction IDs to review.

For EACH transaction, in order:
1. Call get_transaction to read it.
2. Call verify_mandates to validate the AP2 mandate chain (signature, constraints, double-spend).
   HARD GATE: if verify_mandates returns valid=false, constraints_satisfied=false, OR
   double_spend_detected=true, call record_decision immediately with decision="reject" and
   skip the remaining steps for that transaction.
3. Call hybrid_search_similar_frauds to retrieve similar decided precedents.
4. Call detect_fraud_ring on the sender's account_number to check for ring/mule patterns.
5. Weigh the precedents, the ring signal, the amount, and your confidence. Then make EXACTLY
   ONE terminal call for that transaction:
   - You MUST call escalate (do NOT call record_decision yourself) whenever ANY of these holds:
       * a structuring amount ($4,900-$4,999),
       * a high-value amount (>= $50,000) that you would otherwise approve,
       * detect_fraud_ring reports suspicious_patterns, or
       * your confidence is medium (~75-85).
     Give your recommended_decision and a short reason.
   - Otherwise (a clear-cut case matching none of the above), call record_decision (approve/reject).

When you escalate, you will receive the human's decision. Then call record_decision with that
decision, escalated=true, and recommended_decision set to what you had recommended.

Be concise. Move to the next transaction after recording a decision."""

agent = client.beta.agents.create(
    name="MongoDB Atlas fraud reviewer",
    # 이 agent가 어느 cookbook에서 왔는지 Anthropic에 알립니다. 제거해도 안전합니다.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-with-mongodb-atlas"},
    model=MODEL,
    system=SYSTEM,
    tools=TOOLS,
)
environment = client.beta.environments.create(
    name="fraud-review-env", config={"type": "cloud", "networking": {"type": "limited"}}
)
session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Fraud review",
)
print("session:", session.id)
print(f"Watch in Console: https://platform.claude.com/workspaces/default/sessions/{session.id}")

session: sesn_01LbAUC1V36CQS3Sk26eixTc
Watch in Console: https://platform.claude.com/workspaces/default/sessions/sesn_01LbAUC1V36CQS3Sk26eixTc


### human gate로 검토 실행하기

event stream을 열고 대기 중인 transaction queue를 보낸 뒤 `run_gate_loop`로 gate 루프를 진행합니다. data tool 호출은 `HANDLERS`에서 자동 처리되며 `escalate` 호출은 사람을 기다립니다. `AUTO_APPROVE`가 설정되면(예: CI) gate는 결정적으로 처리하고 **초기 데이터의 structuring 사례에서 agent 판정을 덮어써** 다른 인간 판정이 보이도록 합니다. 그렇지 않으면 그 자리에서 입력을 요청합니다.


In [16]:
OVERRIDE_IDS = {"txn-review-struct"}


def resolve_human_decision(recommendation, *, auto_approve, override_ids, txn_id) -> str:
    """사람 검토자 역할을 합니다. CI(auto_approve)에서는 agent에 동의하되, 초기 데이터의
    override_ids에서는 판정을 뒤집어 다른 인간 결정을 보이게 합니다."""
    opposite = "reject" if recommendation == "approve" else "approve"
    if auto_approve and txn_id in set(override_ids):
        return opposite
    return recommendation


def resolver(tool_input):
    txn_id = tool_input.get("transaction_id", "")
    recommended = tool_input.get("recommended_decision", "reject")
    if AUTO_APPROVE:
        decision = resolve_human_decision(
            recommended, auto_approve=True, override_ids=OVERRIDE_IDS, txn_id=txn_id
        )
        flag = "  <-- HUMAN OVERRIDE" if decision != recommended else ""
        print(f"  [human/auto] {txn_id}: agent recommended {recommended} -> human {decision}{flag}")
        return decision
    print(
        f"\n  ESCALATED {txn_id} (agent recommends {recommended}): {tool_input.get('reason', '')}"
    )
    return (
        "approve"
        if input("  Your decision [approve/reject]: ").strip().lower().startswith("a")
        else "reject"
    )


def send_result(custom_tool_use_id, result):
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.custom_tool_result",
                "custom_tool_use_id": custom_tool_use_id,
                "content": [{"type": "text", "text": json.dumps(result)}],
            }
        ],
    )


def run_gate_loop(stream, send, *, handlers, resolver) -> dict:
    """custom tool 호출을 도착하는 대로 처리하며 세션을 완료까지 진행합니다.

    실제 작업의 전체 Path A 왕복 과정입니다. 각 `agent.custom_tool_use` event는 tool과
    입력을 지정하며 세션은 응답할 때까지 `stop_reason == "requires_action"`으로
    idle이 됩니다. `handlers`에서 tool을 찾습니다(호스트 측 `pymongo` 실행 —
    credential은 이 process를 떠나지 않음). 특수 `escalate` tool은 대신 사람
    `resolver`로 보냅니다. 각 결과를 `send`로 돌려보내면 세션이 재개합니다.
    종료 상태 idle(`end_turn`) 또는 termination에서 종료합니다.
    """
    pending: dict[str, Any] = {}
    responded: set[str] = set()
    serviced: list[tuple[str, dict]] = []
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            pending[ev.id] = ev
        elif ev.type == "session.status_idle":
            stop = getattr(ev, "stop_reason", None)
            if stop is None:
                continue
            if stop.type != "requires_action":
                break  # end_turn / retries_exhausted — 실행 완료
            for event_id in stop.event_ids or []:
                if event_id in responded:
                    continue
                call = pending.get(event_id)
                if call is None:
                    continue
                if call.name == "escalate":
                    result = {"human_decision": resolver(call.input)}
                else:
                    handler = handlers.get(call.name)
                    result = (
                        handler(call.input) if handler else {"error": f"unknown tool {call.name}"}
                    )
                responded.add(event_id)
                send(event_id, result)
                serviced.append((call.name, result))
        elif ev.type == "session.status_terminated":
            break
    return {"serviced": serviced, "responded": sorted(responded)}


kickoff = {
    "type": "user.message",
    "content": [
        {"type": "text", "text": "Review these flagged transactions: " + ", ".join(PENDING)}
    ],
}
run_start = datetime.now(UTC)

with client.beta.sessions.events.stream(session_id=session.id) as stream:
    client.beta.sessions.events.send(session_id=session.id, events=[kickoff])
    gate_result = run_gate_loop(stream, send_result, handlers=HANDLERS, resolver=resolver)

wait_for_idle_status(client, session.id)
print(f"\nserviced {len(gate_result['serviced'])} tool calls")


  [verify_mandates] txn-review-clean: pass

  [verify_mandates] txn-review-fraud: pass

  [verify_mandates] txn-review-struct: pass

  [verify_mandates] txn-review-high: pass

  [verify_mandates] txn-review-ring: pass

  ESCALATED txn-review-struct (agent recommends reject): Structuring indicator: cash deposit of $4,950 just under $5,000 CTR threshold, matching precedent (txn-struct-01, txn-struct-02) showing pattern of deliberate sub-threshold deposits from same account. Amount in structuring range per policy. Requires human review.


  Your decision [approve/reject]:  reject



  ESCALATED txn-review-high (agent recommends approve): High-value wire of $75,000 from Northwind Foods to Pacific Logistics for bulk shipment. Legitimate business transaction matching precedent txn-high-01 ($120,000 supplier wire, approved). Mandate valid, no fraud signals. Escalation required per policy for high-value transactions ≥ $50,000.


  Your decision [approve/reject]:  approve



  ESCALATED txn-review-ring (agent recommends reject): Money-mule ring detected. Sender ACC-RING-A (Quartz Trading) is part of a 4-account network with circular flows. Precedent txn-ring-01 (same sender/recipient, $920, rejected) shows identical pattern. Fraud ring analysis confirms suspicious_patterns, circular_flow. Requires human review for definitive ring-case determination.


  Your decision [approve/reject]:  reject



serviced 28 tool calls


### 결정과 audit trail 검토하기

MongoDB에서 결정과 audit trail을 다시 읽습니다. decision과 audit collection은 **append-only**이므로 이번 실행 범위만 읽습니다. 마지막 assertion은 **self-check**를 수행합니다. 예상한 처리 경로 구성 비율을 재현하지 못하는 재실행은 명확하게 실패합니다.

human override 수는 gate를 어떻게 처리했는지 나타냅니다. `AUTO_APPROVE`에서는 1이며(`txn-review-struct`의 초기 override 표시), interactive 실행에서는 모든 escalation에서 agent 추천에 동의하면 0입니다. 둘 다 예상된 결과입니다.


In [17]:
from collections import Counter

decisions = list(db["transaction_decisions"].find({"created_at": {"$gte": run_start}}, {"_id": 0}))
audits = list(db["audit_events"].find({"timestamp": {"$gte": run_start}}, {"_id": 0}))

print("Decisions by lane:")
lanes = Counter()
for d in decisions:
    txn = coll.find_one({"transaction_id": d["transaction_id"]})
    lanes[f"{(txn['lane'] if txn else '?')} -> {d['decision']}"] += 1
for k, v in sorted(lanes.items()):
    print(f"  {k}: {v}")

overrides = [
    a
    for a in audits
    if a["event_type"] == "escalated_to_human"
    and a["event_data"].get("human_decision") != a["event_data"].get("recommended_decision")
]
print(
    f"\nescalated_to_human events: {sum(1 for a in audits if a['event_type'] == 'escalated_to_human')}"
)
print(f"human overrides (verdict != agent recommendation): {len(overrides)}")
for a in overrides:
    ed = a["event_data"]
    print(
        f"  {a['transaction_id']}: agent {ed['recommended_decision']} -> human {ed['human_decision']}"
    )

# self-check — 이번 실행이 예상한 결과를 만들지 않았다면 명확하게 실패합니다.
decided_lanes = {coll.find_one({"transaction_id": d["transaction_id"]})["lane"] for d in decisions}
assert decided_lanes == {"clean_approve", "clear_reject", "high_value", "ring", "structuring"}, (
    f"expected all 5 lanes decided, saw {sorted(decided_lanes)}"
)
assert len(decisions) == len(PENDING), f"expected {len(PENDING)} decisions, got {len(decisions)}"
assert sum(1 for a in audits if a["event_type"] == "escalated_to_human") >= 1, (
    "expected >=1 escalation"
)
if AUTO_APPROVE:
    assert overrides, "AUTO_APPROVE run should show the seeded override on txn-review-struct"
print("\nself-check OK")

Decisions by lane:
  clean_approve -> approve: 1
  clear_reject -> reject: 1
  high_value -> approve: 1
  ring -> reject: 1
  structuring -> reject: 1

escalated_to_human events: 3
human overrides (verdict != agent recommendation): 0

self-check OK


## 4. system of record와 audit 기반으로서의 MongoDB Atlas

한발 물러나 실행 상태가 어디에 있는지 보세요. agent가 한 모든 작업이 retrieval에 사용한 같은 cluster에 지속적으로 저장됩니다. 쓰기 측에서 얻는 단일 engine의 이점입니다. `transactions`는 operational record이며(각 사례가 `pending` → `approved` 또는 `rejected`로 진행), `transaction_decisions`는 immutable 판정을 보관하고, `audit_events`는 append-only trail이며, `mandate_receipts`는 double-spend 감지를 지원합니다. 사례 하나의 전체 과정은 다음과 같습니다.


In [18]:
case = "txn-review-struct"
txn = coll.find_one({"transaction_id": case}, {"_id": 0, "embedding": 0})
print(f"operational record: {case}  lane={txn['lane']}  status={txn['status']}")

decision = db["transaction_decisions"].find_one(
    {"transaction_id": case, "created_at": {"$gte": run_start}}, {"_id": 0}
)
print(
    f"decision record:    {decision['decision']}  reviewed_by={decision['reviewed_by']}  confidence={decision['confidence_score']}"
)

print("audit trail (append-only):")
for a in (
    db["audit_events"]
    .find({"transaction_id": case, "timestamp": {"$gte": run_start}}, {"_id": 0})
    .sort("timestamp", 1)
):
    extra = ""
    if a["event_type"] == "escalated_to_human":
        ed = a["event_data"]
        extra = f"  (agent recommended {ed['recommended_decision']}, human decided {ed['human_decision']})"
    print(f"  {a['timestamp']:%H:%M:%S}  {a['event_type']}  severity={a['severity']}{extra}")

txn_mandate = coll.find_one({"transaction_id": case}, {"mandate_id": 1, "agent_pk": 1})
if txn_mandate and txn_mandate.get("mandate_id"):
    receipt = db["mandate_receipts"].find_one({"mandate_id": txn_mandate["mandate_id"]}, {"_id": 0})
    if receipt:
        print(
            f"mandate receipt:    mandate_id={receipt['mandate_id'][:12]}…  decision={receipt['decision']}"
        )

operational record: txn-review-struct  lane=structuring  status=rejected
decision record:    reject  reviewed_by=human  confidence=0.9
audit trail (append-only):
  13:49:22  escalated_to_human  severity=warning  (agent recommended reject, human decided reject)


**production webhook(안내이며 여기서는 실행하지 않음):** 위 streaming 루프는 개발에 적합하지만 재시작 후 유지되지 않습니다. production에서는 `session.status_idled` webhook을 등록하세요. handler는 검토자용 queue에 사례를 넣고 나중에 `user.custom_tool_result`를 POST합니다. **data tool handler는 동일**하며 trigger만 바뀝니다. [`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb)를 참고하세요.


In [19]:
client.beta.sessions.archive(session.id)
client.beta.environments.archive(environment.id)
# agent는 실행 사이에 재사용할 수 있습니다. archive는 선택 사항이며 영구적입니다.
mongo.close()
print("archived session + environment; closed MongoDB connection")

archived session + environment; closed MongoDB connection


## 5. 되짚어 보기

MongoDB Atlas를 Claude Managed Agent에 연결하고 한 번의 구축으로 연결부터 system of record까지 구현했습니다.

- **Connect** — credential이 안전한 data path 세 개(host 측 custom tool, self-hosted sandbox, self-hosted MCP). MongoDB secret은 항상 경계의 당신 측에 있습니다.
- **Retrieve** — 2절에 직접 정의하여 가져올 수 있는 pipeline builder 형태의 vector, full-text, hybrid RRF, graph traversal.
- **Gate** — CMA의 native `requires_action` human pause로 위험한 결정에 조건 두기.
- **Persist** — agent가 retrieval하는 같은 cluster에 결정, append-only audit trail, AP2 receipt 보존하기.

integration 인터페이스는 작았습니다. custom tool 뒤의 몇 줄 `pymongo`입니다. collection과 tool을 바꾸면 같은 형태가 모든 agent에 MongoDB 근거를 제공합니다. retrieval builder, tool handler, gate 루프 모두 여기 notebook에 있으며 [`mongodb_on_cma/`](mongodb_on_cma/) package는 설정 boilerplate만 보관합니다.
